# Getting Started with cleanskate

Figure skating data gets interesting fast: events, segments, standings, protocol sheets, element calls, judge marks. `cleanskate` tries to put all of that into plain pandas data frames so you can start asking questions without first building a scraper.

This notebook is a quick tour of the package. We'll load the main tables, peek at the shape of the dataset, make a few small summaries, and then drill from a result row down into the underlying protocol details.


## Load the dataset

Everything starts with a `Dataset` handle. The first call may download hosted files into your local cache; after that, the same tables load quickly from disk.


In [ ]:
import matplotlib as mpl
import numpy as np
import pandas as pd
import seaborn as sns

from cleanskate import Dataset

mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False

ds = Dataset(version="latest")


Let's load the core tables once at the top. In a notebook, this keeps the rest of the work feeling like ordinary pandas.


In [ ]:
events = ds.load_events()
segments = ds.load_segments()
standings = ds.load_standings()
results = ds.load_results()
elements = ds.load_elements()
components = ds.load_program_components()


A quick row-count check is a nice first sanity pass. It tells us what kind of scale we are working with before we start slicing.


In [ ]:
pd.DataFrame(
    {
        "table": ["events", "segments", "standings", "results", "elements", "program_components"],
        "rows": [len(events), len(segments), len(standings), len(results), len(elements), len(components)],
    }
)


## Start with events and segments

Before slicing into jumps or scores, it helps to see what competitions and segments are in the snapshot. The event table gives the broad coverage; the segment table shows the actual Men/Women/Pairs/Ice Dance units you will usually analyze.


In [ ]:
events.sort_values(
    ["season", "event_series", "event_label"],
    ascending=[False, True, True],
).head(10)


The segment table lets us check how much material is available by level and discipline.


In [ ]:
segments.groupby(["event_level", "discipline"], dropna=False).size().unstack(fill_value=0)


## Use loader filters

The loader methods take readable filters like `season`, `event_series`, `event_level`, `discipline`, and `segment_label`. Passing a list means "any of these," which keeps notebook code pleasantly direct.


In [ ]:
senior_worlds_results = ds.load_results(
    event_series="Worlds",
    event_level="Senior",
    discipline=["Men", "Women"],
)

senior_worlds_results.head()


That same pattern works on every public loader. The point is to keep common subsetting close to the data-loading step, so the rest of the notebook can stay focused on the analysis.


## Compare segment scores in final standings

A simple place to start is the relationship between first and second segment scores. Final standings already carry both segment scores, so we can quickly see how short-program and free-skate results move together across disciplines.


In [ ]:
combined_standings = standings[
    standings["standing_type"].eq("Final")
    & standings["segment_1_score"].notna()
    & standings["segment_2_score"].notna()
].copy()

combined_standings[[
    "event_label",
    "discipline",
    "rank",
    "name",
    "segment_1_score",
    "segment_2_score",
]].head()


Now we can plot the relationship. Each panel is one discipline, and the small `r` annotation is just the within-panel correlation.


In [ ]:
def annotate_corr(data, **kws):
    subset = data[["segment_1_score", "segment_2_score"]].dropna()
    ax = mpl.pyplot.gca()
    if len(subset) > 1 and subset["segment_1_score"].nunique() > 1 and subset["segment_2_score"].nunique() > 1:
        corr = np.corrcoef(subset["segment_1_score"], subset["segment_2_score"])[0, 1]
        label = f"r = {corr:.2f}"
    else:
        label = "r = NA"
    ax.text(0.05, 0.95, label, transform=ax.transAxes, ha="left", va="top")

g = sns.relplot(
    data=combined_standings,
    x="segment_1_score",
    y="segment_2_score",
    col="discipline",
    col_wrap=4,
    height=3.2,
    aspect=1,
    alpha=0.55,
)
g.map_dataframe(annotate_corr)
g.set_axis_labels("Segment 1 score", "Segment 2 score")


## Find strong triple axel attempts

Now let's move from event-level results into protocol rows. The elements table has one row per scored element, so a question like "who tends to get rewarded on triple axels?" is just a filter plus a groupby.


In [ ]:
triple_axels = ds.load_elements(attempt_code="3A")

triple_axels[[
    "event_label",
    "segment_label",
    "name",
    "element_code",
    "goe",
    "panel_score",
    "clean_element",
]].head()


From there, we can summarize skaters with enough attempts to make the averages somewhat meaningful.


In [ ]:
triple_axel_summary = (
    triple_axels.groupby("name", as_index=False)
    .agg(
        attempts=("goe", "size"),
        mean_goe=("goe", "mean"),
        mean_panel_score=("panel_score", "mean"),
        clean_rate=("clean_element", "mean"),
        fall_rate=("fall", "mean"),
    )
    .query("attempts >= 10")
    .sort_values("mean_goe", ascending=False)
)

triple_axel_summary.head(10)


## Profile one skater's jump scoring

Another way to use the elements table is to zoom in on one skater and ask what their jumps are worth in practice. Let's look at Ilia Malinin as an example.


In [ ]:
senior_jumps = ds.load_elements(
    event_level="Senior",
    element_family=["Jump", "Jump Combo", "Jump Sequence"],
)

ilia_jumps = senior_jumps[senior_jumps["name"].eq("Ilia MALININ")].copy()

ilia_jumps[["event_label", "segment_label", "element_code", "attempt_code", "goe", "panel_score"]].head()


First, a compact table: what does he attempt often enough to summarize, and what does each attempt tend to be worth?


In [ ]:
ilia_summary = (
    ilia_jumps.groupby(["element_family", "attempt_code"], as_index=False)
    .agg(
        attempts=("goe", "size"),
        mean_goe=("goe", "mean"),
        mean_panel_score=("panel_score", "mean"),
        clean_rate=("clean_element", "mean"),
        fall_rate=("fall", "mean"),
    )
    .query("attempts >= 5")
    .sort_values("mean_panel_score", ascending=False)
)

ilia_summary


Then, the same idea as a distribution. The table gives the average; the plot shows the spread.


In [ ]:
attempt_order = ilia_summary["attempt_code"].tolist()
plot_data = ilia_jumps[ilia_jumps["attempt_code"].isin(attempt_order)].copy()


In [ ]:
fig, axes = mpl.pyplot.subplots(1, 2, figsize=(13, 4), sharex=True)

sns.boxplot(
    data=plot_data,
    x="attempt_code",
    y="panel_score",
    order=attempt_order,
    width=0.6,
    fliersize=3,
    linewidth=1.1,
    color="#899BB8",
    ax=axes[0],
)
axes[0].set_xlabel("Attempt")
axes[0].set_ylabel("Panel score")
axes[0].tick_params(axis="x", rotation=30)

sns.boxplot(
    data=plot_data,
    x="attempt_code",
    y="goe",
    order=attempt_order,
    width=0.6,
    fliersize=3,
    linewidth=1.1,
    color="#899BB8",
    ax=axes[1],
)
axes[1].set_xlabel("Attempt")
axes[1].set_ylabel("GOE")
axes[1].tick_params(axis="x", rotation=30)
fig.tight_layout()


## Summarize technical calls

Scores only tell part of the story. A skater's protocol also records what the technical panel called: quarter landings, underrotations, downgrades, edge calls, invalid elements, and falls. Those columns make it easy to ask what kinds of problems show up, not just how many points were earned.


In [ ]:
call_rates = senior_jumps.assign(
    rotation_issue=lambda frame: frame[["call_quarter", "call_underrotated", "call_downgraded"]].any(axis=1),
    edge_issue=lambda frame: frame[["call_edge_attention", "call_wrong_edge"]].any(axis=1),
)

call_rates[[
    "name",
    "element_code",
    "rotation_issue",
    "edge_issue",
    "fall",
    "invalid_element",
]].head()


Once the individual call flags are collapsed into analysis-friendly groups, a skater-level summary is just another groupby.


In [ ]:
skater_call_rates = (
    call_rates.groupby("name", as_index=False)
    .agg(
        attempts=("goe", "size"),
        rotation_issue_rate=("rotation_issue", "mean"),
        edge_issue_rate=("edge_issue", "mean"),
        fall_rate=("fall", "mean"),
        invalid_rate=("invalid_element", "mean"),
    )
    .query("attempts >= 50")
    .sort_values("rotation_issue_rate", ascending=False)
)

skater_call_rates.head(10)


## Drill into protocol details for one result

The summary tables and protocol tables are connected. If you start from a visible result row, `cleanskate` can resolve the underlying result ID and pull back the elements and program components for that exact skate.


In [ ]:
sample_result = results.sort_values("total_segment_score", ascending=False).iloc[0]

sample_result[["event_label", "segment_label", "name", "noc", "total_segment_score"]]


Here are the elements from that skate.


In [ ]:
sample_elements = ds.load_elements_for_result(sample_result)

sample_elements[["element_number", "element_code", "base_value", "goe", "panel_score", "info_flags"]].head(12)


And here are the program components attached to the same result row.


In [ ]:
sample_components = ds.load_program_components_for_result(sample_result)

sample_components[["component_name", "factor", "average", "judge_scores"]]


## Next steps

That is the basic rhythm: load a table, filter to the skating question you care about, then stay in pandas. From here you can get more specific: a season, an event series, a discipline, an element family, a skater, a panel, or a single protocol sheet.
